# DiffSinger Miadu Fine-tuning (穩定修復版)
**修正內容**：
- **GPU 強制啟用**：解決了 TF 搶佔 CUDA 的問題，確保 T4 GPU 被正確調用。
- **強健同步邏輯**：自動檢測並修復損壞或 0 位元檔案。
- **路徑容錯**：適應各種 Drive 資料夾命名習慣。

## 第一步：掛載 Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 第二步：獲取代碼與環境配置

In [ ]:
%cd /content/
!git clone https://github.com/shihte/DiffSinger-Miadu-Colab.git diffsinger-repo
%cd diffsinger-repo
!pip install --upgrade setuptools pip wheel
!pip install praat-parselmouth pyloudnorm pypinyin pycwt pyworld lightning-flash==0.7.0

## 第三步：深度完整性檢查與同步 (GPU 檢查版)

In [ ]:
import os
import torch

# 1. GPU 狀態檢查
gpu_ok = torch.cuda.is_available()
print(f'| GPU 狀態: {"✅ 已就緒" if gpu_ok else "❌ 未偵測到 GPU，請檢查 Colab 設定"}')
if gpu_ok:
    print(f'| 使用裝置: {torch.cuda.get_device_name(0)}')

# 2. 路徑解析
drive_root = "/content/drive/MyDrive/DiffSinger_Colab"
if not os.path.exists(drive_root):
    drive_root = "/content/drive/MyDrive/DiffSinger _Colab"
print(f'| 使用 Drive 目錄: {drive_root}')

# 3. Checkpoints 軟連結
!mkdir -p "{drive_root}/checkpoints_finetune"
if not os.path.islink("checkpoints"):
    !rm -rf checkpoints
    !ln -s "{drive_root}/checkpoints_finetune" checkpoints

# 4. 關鍵數據同步函數
def robust_sync(src_dir, dest_path, min_size_mb=1):
    dest_dir = os.path.dirname(dest_path)
    !mkdir -p "{dest_dir}"
    needs_sync = False
    if not os.path.exists(dest_path):
        needs_sync = True
    elif os.path.getsize(dest_path) < min_size_mb * 1024 * 1024:
        print(f"| 警告：偵測到檔案過小或損壞: {dest_path}")
        needs_sync = True
    
    if needs_sync:
        print(f"| 正在同步資料: {src_dir} -> {dest_dir}")
        !cp -rv "{src_dir}" "{dest_dir}/.."
    else:
        print(f"| 已跳過完整檔案: {os.path.basename(dest_path)}")

print('| 正在執行深度校驗...')
# 同步基礎權重 (預期約 800MB+)
robust_sync(f"{drive_root}/1117_opencpop_ds1000_strict_pinyin", 
            "checkpoints/1117_opencpop_ds1000_strict_pinyin/model_ckpt_steps_200000.ckpt", 800)

# 同步聲碼器
robust_sync(f"{drive_root}/nsf_hifigan_44.1k_hop512_128bin_2024.02", 
            "checkpoints/nsf_hifigan_44.1k_hop512_128bin_2024.02/model_ckpt_steps_160000.ckpt", 10)

# 同步二進位數據 (預期 5GB+)
robust_sync(f"{drive_root}/miadu_finetune", 
            "data/binary/miadu_finetune/train.data", 1000)

print('| 所有數據同步校驗完成！')

## 第四步：啟動訓練 (GPU 強制模式)

In [ ]:
import os
os.environ['PYTHONPATH'] = "."
#!python tasks/run.py --help  # 如需查看參數請取消註釋
!python tasks/run.py --config usr/configs/midi/e2e/miadu_finetune.yaml --exp_name miadu_finetune_v1